# 🚀 Skrip Scraping Data X (Twitter) Wacana ASEAN - Google Colab

Notebook ini dirancang untuk menjalankan *scraping* data tuitan X terkait isu Papua & Identitas Melanesia di wilayah **ASEAN (non-Indonesia)** langsung di cloud Google Colab.

### 📌 Cara Menggunakan:
1. Jalankan **Sel 1** untuk menginstal library yang dibutuhkan.
2. Jalankan **Sel 2**, lalu masukkan nilai `auth_token` dan `ct0` dari cookie X akun dummy Anda saat diminta.
3. Setelah selesai, file CSV hasil scraping akan **otomatis terunduh** ke komputer Anda!

In [1]:
# SEL 1: Install Dependencies
!pip install twikit pandas nest_asyncio beautifulsoup4

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 20.4 MB/s eta 0:00:00
  Created wheel for pyjsparser: filename=pyjsparser-2.7.1-py3-none-any.whl size=25982 sha256=1f94b096cb0c0bddb8f67d9080a179ef1a59db4f53d30e7e4df086c6e32f0b8e
  Stored in directory: /root/.cache/pip/wheels/18/da/21/514147d1860b444fc1a2581d6aefe0e03e56564433d0ca8208
Successfully built pyjsparser


In [2]:
# SEL 2: Skrip Utama Scraping dengan Twikit Anti-Crash & 404 Patch
import asyncio
import os
import random
import sys
import pandas as pd
import nest_asyncio
import bs4
import twikit
from twikit import Client
import twikit.x_client_transaction.transaction as t
import twikit.client.gql as g
import twikit.user as u

nest_asyncio.apply()

# Patch twikit core (Perbaikan 404, KeyError 'urls', & ClientTransaction)
def apply_all_twikit_patches():
    async def safe_init(self, session, headers):
        try:
            home_page_response = await t.handle_x_migration(session, headers)
            self.home_page_response = self.validate_response(home_page_response)
        except Exception:
            self.home_page_response = bs4.BeautifulSoup(
                '<html><head><meta name="twitter-site-verification" content="mentU+1yPz+IcNS+RaF+b"/></head><body></body></html>',
                'html.parser'
            )

        try:
            self.DEFAULT_ROW_INDEX, self.DEFAULT_KEY_BYTES_INDICES = await self.get_indices(self.home_page_response, session, headers)
        except Exception:
            self.DEFAULT_ROW_INDEX = 2
            self.DEFAULT_KEY_BYTES_INDICES = [12, 14, 7]

        try:
            self.key = self.get_key(response=self.home_page_response)
        except Exception:
            self.key = "mentU+1yPz+IcNS+RaF+b"

        self.key_bytes = self.get_key_bytes(key=self.key)
        try:
            self.animation_key = self.get_animation_key(key_bytes=self.key_bytes, response=self.home_page_response)
        except Exception:
            self.animation_key = "00000000"

    async def patched_search_timeline(self, query: str, product: str, count: int, cursor: str | None):
        variables = {
            'rawQuery': query,
            'count': count,
            'querySource': 'typed_query',
            'product': product
        }
        if cursor is not None:
            variables['cursor'] = cursor
        return await self.gql_post(g.Endpoint.SEARCH_TIMELINE, variables, g.FEATURES)

    def safe_user_init(self, client, data):
        self._client = client
        legacy = data.get('legacy', {})

        self.id = data.get('rest_id', '')
        self.created_at = legacy.get('created_at', '')
        self.name = legacy.get('name', '')
        self.screen_name = legacy.get('screen_name', '')
        self.profile_image_url = legacy.get('profile_image_url_https', '')
        self.profile_banner_url = legacy.get('profile_banner_url')
        self.url = legacy.get('url')
        self.location = legacy.get('location', '')
        self.description = legacy.get('description', '')

        entities = legacy.get('entities', {})
        desc_entities = entities.get('description', {}) if isinstance(entities, dict) else {}
        self.description_urls = desc_entities.get('urls', []) if isinstance(desc_entities, dict) else []

        url_entities = entities.get('url', {}) if isinstance(entities, dict) else {}
        self.urls = url_entities.get('urls', []) if isinstance(url_entities, dict) else []

        self.pinned_tweet_ids = legacy.get('pinned_tweet_ids_str', [])
        self.is_blue_verified = data.get('is_blue_verified', False)
        self.verified = legacy.get('verified', False)
        self.possibly_sensitive = legacy.get('possibly_sensitive', False)
        self.can_dm = legacy.get('can_dm', False)
        self.can_media_tag = legacy.get('can_media_tag', False)
        self.want_retweets = legacy.get('want_retweets', False)
        self.default_profile = legacy.get('default_profile', False)
        self.default_profile_image = legacy.get('default_profile_image', False)
        self.has_custom_timelines = legacy.get('has_custom_timelines', False)
        self.followers_count = legacy.get('followers_count', 0)
        self.fast_followers_count = legacy.get('fast_followers_count', 0)
        self.normal_followers_count = legacy.get('normal_followers_count', 0)
        self.following_count = legacy.get('friends_count', 0)
        self.favourites_count = legacy.get('favourites_count', 0)
        self.listed_count = legacy.get('listed_count', 0)
        self.media_count = legacy.get('media_count', 0)
        self.statuses_count = legacy.get('statuses_count', 0)
        self.is_translator = legacy.get('is_translator', False)
        self.translator_type = legacy.get('translator_type', '')
        self.withheld_in_countries = legacy.get('withheld_in_countries', [])
        self.protected = legacy.get('protected', False)

    t.ClientTransaction.init = safe_init
    g.GQLClient.search_timeline = patched_search_timeline
    u.User.__init__ = safe_user_init

apply_all_twikit_patches()

OUTPUT_CSV_PATH = "asean_papua_x_tweets.csv"
MAX_TWEETS_TO_COLLECT = 500
MIN_DELAY = 3.0
MAX_DELAY = 7.0

SEARCH_QUERIES = [
    '("West Papua" OR "Papua" OR "Melanesia" OR "MSG") ("sovereignty" OR "human rights" OR "diplomacy" OR "identity" OR "autonomy")',
    '("West Papua" OR "Papua" OR "Melanesian") ("independence" OR "conflict" OR "Pacific" OR "ASEAN")',
    '("Isu Papua" OR "Rakyat Papua" OR "Identiti Melanesia")'
]

ASEAN_LOCATION_KEYWORDS = {
    "Malaysia": ["malaysia", "kuala lumpur", "kl", "penang", "johor", "selangor", "cyberjaya", "my"],
    "Singapore": ["singapore", "sg"],
    "Thailand": ["thailand", "bangkok", "chiang mai", "phuket", "th"],
    "Philippines": ["philippines", "manila", "quezon city", "cebu", "ph"],
    "Vietnam": ["vietnam", "hanoi", "ho chi minh", "saigon", "vn"],
    "Myanmar": ["myanmar", "yangon", "mandalay", "burma", "mm"],
    "Cambodia": ["cambodia", "phnom penh", "kh"],
    "Laos": ["laos", "vientiane", "la"],
    "Brunei": ["brunei", "bandar seri begawan", "bn"]
}

EXCLUDE_LOCATION_KEYWORDS = [
    "indonesia", "jakarta", "surabaya", "bandung", "yogyakarta", "jogja",
    "bali", "medan", "semarang", "makassar", "palembang", "depok", "tangerang",
    "bekasi", "bogor", "jayapura", "idn", "id"
]

def is_indonesia_location(user_location: str) -> bool:
    if not user_location:
        return False
    loc_lower = user_location.lower().strip()
    for ex_key in EXCLUDE_LOCATION_KEYWORDS:
        if ex_key in loc_lower:
            return True
    return False

def detect_asean_country(user_location: str) -> str:
    if not user_location:
        return "ASEAN (Lainnya/Tidak Spesifik)"

    loc_lower = user_location.lower().strip()
    if is_indonesia_location(loc_lower):
        return "INDONESIA_EXCLUDED"

    for country, keywords in ASEAN_LOCATION_KEYWORDS.items():
        for kw in keywords:
            if kw in loc_lower:
                return country

    return "ASEAN (Lainnya/Non-ID)"

def load_existing_tweet_ids(filepath: str) -> set:
    if os.path.exists(filepath):
        try:
            df = pd.read_csv(filepath, usecols=["tweet_id"])
            return set(df["tweet_id"].astype(str).tolist())
        except Exception:
            return set()
    return set()

def save_to_csv(tweets_list: list, filepath: str):
    df_new = pd.DataFrame(tweets_list)
    file_exists = os.path.exists(filepath)
    df_new.to_csv(filepath, mode='a' if file_exists else 'w', index=False, header=not file_exists, encoding='utf-8-sig')
    print(f"[CHECKPOINT] {len(tweets_list)} tuitan disimpan ke CSV.")

async def main():
    print("="*70)
    print(" SCRAPING DATA X (TWITTER) WACANA ASEAN - GOOGLE COLAB")
    print("="*70)

    auth_token = input("Masukkan auth_token dari cookie X Anda: ").strip()
    ct0 = input("Masukkan ct0 dari cookie X Anda: ").strip()

    if not auth_token or not ct0:
        print("[ERROR] auth_token dan ct0 tidak boleh kosong!")
        return

    print("\n[INFO] Menginisialisasi Klien X...")
    client = Client(language='en-US')
    cookies = {"auth_token": auth_token, "ct0": ct0}
    client.set_cookies(cookies)
    print("[SUCCESS] Cookies berhasil dipasang!")

    collected_tweets = []
    seen_ids = load_existing_tweet_ids(OUTPUT_CSV_PATH)
    total_collected_this_run = 0

    for query in SEARCH_QUERIES:
        if total_collected_this_run >= MAX_TWEETS_TO_COLLECT:
            break

        print(f"\n[QUERY] Running: {query}")
        try:
            tweets = await client.search_tweet(query, product='Top', count=20)
            page_count = 0

            while tweets and total_collected_this_run < MAX_TWEETS_TO_COLLECT:
                page_count += 1
                print(f" -> Halaman {page_count} ({len(tweets)} tuitan ditemukan)...")

                for tweet in tweets:
                    tweet_id = str(tweet.id)
                    if tweet_id in seen_ids:
                        continue

                    user = tweet.user
                    user_location = getattr(user, 'location', '') or ''
                    country_status = detect_asean_country(user_location)
                    if country_status == "INDONESIA_EXCLUDED":
                        continue

                    full_text = getattr(tweet, 'full_text', '') or getattr(tweet, 'text', '')
                    cleaned_text = " ".join(full_text.split())

                    tweet_data = {
                        "tweet_id": tweet_id,
                        "created_at": str(tweet.created_at),
                        "username": user.screen_name,
                        "user_name": user.name,
                        "user_location": user_location,
                        "detected_country": country_status,
                        "full_text": cleaned_text,
                        "language": getattr(tweet, 'lang', 'unknown'),
                        "retweet_count": getattr(tweet, 'retweet_count', 0),
                        "like_count": getattr(tweet, 'favorite_count', 0),
                        "reply_count": getattr(tweet, 'reply_count', 0),
                        "tweet_url": f"https://x.com/{user.screen_name}/status/{tweet_id}"
                    }

                    collected_tweets.append(tweet_data)
                    seen_ids.add(tweet_id)
                    total_collected_this_run += 1

                if len(collected_tweets) >= 20:
                    save_to_csv(collected_tweets, OUTPUT_CSV_PATH)
                    collected_tweets = []

                delay = random.uniform(MIN_DELAY, MAX_DELAY)
                await asyncio.sleep(delay)

                try:
                    tweets = await tweets.next()
                except Exception as e:
                    print(f" [INFO] Akhir batch hasil pencarian: {e}")
                    break

        except Exception as e:
            print(f"[ERROR] Kendala query '{query}': {e}")
            await asyncio.sleep(5)

    if collected_tweets:
        save_to_csv(collected_tweets, OUTPUT_CSV_PATH)

    print("\n" + "="*70)
    print(f"[SELESAI] Total tuitan baru berhasil disimpan: {total_collected_this_run}")
    print("="*70 + "\n")

    try:
        from google.colab import files
        print("[INFO] Mengunduh file CSV hasil scraping ke komputer Anda...")
        files.download(OUTPUT_CSV_PATH)
    except Exception:
        pass

await main()

 SCRAPING DATA X (TWITTER) WACANA ASEAN - GOOGLE COLAB
Masukkan auth_token dari cookie X Anda: a884a7955d09028f8255bc74216cf74307677cf5
Masukkan ct0 dari cookie X Anda: a49fde228180ec3efe35fcaa9ab2978dfa02bce8ec7db935db0c171ba3ff9c1be5b72762fefa0584a3205b6aad87b1a50c546abe11115c872eea8bd13faae8d785ef36a24a61500681a3b224ce29a268

[INFO] Menginisialisasi Klien X...
[SUCCESS] Cookies berhasil dipasang!

[QUERY] Running: ("West Papua" OR "Papua" OR "Melanesia" OR "MSG") ("sovereignty" OR "human rights" OR "diplomacy" OR "identity" OR "autonomy")
 -> Halaman 1 (20 tuitan ditemukan)...
 -> Halaman 2 (19 tuitan ditemukan)...
[CHECKPOINT] 31 tuitan disimpan ke CSV.
 -> Halaman 3 (20 tuitan ditemukan)...
 -> Halaman 4 (16 tuitan ditemukan)...
[CHECKPOINT] 33 tuitan disimpan ke CSV.
 -> Halaman 5 (15 tuitan ditemukan)...
 -> Halaman 6 (18 tuitan ditemukan)...
[CHECKPOINT] 33 tuitan disimpan ke CSV.
 -> Halaman 7 (16 tuitan ditemukan)...
 -> Halaman 8 (16 tuitan ditemukan)...
[CHECKPOINT] 32 tuit

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>